In [1]:
import yfinance as yf
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

# ------------------------------------
# 1️⃣ Define Companies
# ------------------------------------
tickers = ["RELIANCE.NS", "TCS.NS", "INFY.NS"]

all_data = []

# ------------------------------------
# 2️⃣ Fetch & Process Data
# ------------------------------------
for ticker in tickers:

    print(f"Fetching data for {ticker}...")

    df = yf.download(
        ticker,
        period="7d",
        interval="5m",
        auto_adjust=False
    )

    # Fix MultiIndex issue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Convert UTC → IST
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC").tz_convert("Asia/Kolkata")
    else:
        df.index = df.index.tz_convert("Asia/Kolkata")

    df.reset_index(inplace=True)

    df.rename(columns={
        "Datetime": "timestamp",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume"
    }, inplace=True)

    df["company_name"] = ticker

    # ------------------------------------
    # Financial Calculations
    # ------------------------------------

    # Return (NaN replaced with 0)
    df["return"] = df["close"].pct_change().fillna(0)

    # Moving averages with NaN replaced
    df["ma_20"] = df["close"].rolling(window=20).mean().fillna(0)
    df["ma_50"] = df["close"].rolling(window=50).mean().fillna(0)

    # ------------------------------------
    # Keep Required Columns
    # ------------------------------------
    df = df[
        [
            "company_name",
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "return",
            "ma_20",
            "ma_50",
        ]
    ]

    all_data.append(df)

# ------------------------------------
# 3️⃣ Combine All Companies
# ------------------------------------
final_df = pd.concat(all_data, ignore_index=True)

# Replace infinite values
final_df.replace([float("inf"), -float("inf")], None, inplace=True)

# Replace remaining NaN with None
final_df = final_df.where(pd.notnull(final_df), None)

records = final_df.values.tolist()

print("Column count:", len(final_df.columns))
print("Values per row:", len(records[0]))

# ------------------------------------
# 4️⃣ Connect PostgreSQL
# ------------------------------------
conn = psycopg2.connect(
    dbname="y_finance",
    user="postgres",
    password="postgres",   # change if needed
    host="localhost",
    port="5432"
)

cur = conn.cursor()

# ------------------------------------
# 5️⃣ Drop Old Table
# ------------------------------------
cur.execute("DROP TABLE IF EXISTS stock_data;")
conn.commit()

# ------------------------------------
# 6️⃣ Create Table
# ------------------------------------
create_table_query = """
CREATE TABLE stock_data (
    company_name TEXT,
    timestamp TIMESTAMP,
    open DOUBLE PRECISION,
    high DOUBLE PRECISION,
    low DOUBLE PRECISION,
    close DOUBLE PRECISION,
    volume BIGINT,
    return DOUBLE PRECISION,
    ma_20 DOUBLE PRECISION,
    ma_50 DOUBLE PRECISION,
    PRIMARY KEY (company_name, timestamp)
);
"""

cur.execute(create_table_query)
conn.commit()

# ------------------------------------
# 7️⃣ Insert Data
# ------------------------------------
insert_query = """
INSERT INTO stock_data
(company_name, timestamp, open, high, low, close,
 volume, return, ma_20, ma_50)
VALUES (%s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s);
"""

execute_batch(cur, insert_query, records, page_size=1000)

conn.commit()

cur.close()
conn.close()

print("✅ Data inserted successfully for 3 companies.")

Fetching data for RELIANCE.NS...


[*********************100%***********************]  1 of 1 completed


Fetching data for TCS.NS...


[*********************100%***********************]  1 of 1 completed


Fetching data for INFY.NS...


[*********************100%***********************]  1 of 1 completed


Column count: 10
Values per row: 10
✅ Data inserted successfully for 3 companies.


In [2]:
final_df

Price,company_name,timestamp,open,high,low,close,volume,return,ma_20,ma_50
0,RELIANCE.NS,2026-03-13 09:15:00+05:30,1386.699951,1397.500000,1384.199951,1394.300049,0,0.000000,0.000000,0.000000
1,RELIANCE.NS,2026-03-13 09:20:00+05:30,1393.800049,1396.599976,1391.000000,1392.800049,201134,-0.001076,0.000000,0.000000
2,RELIANCE.NS,2026-03-13 09:25:00+05:30,1392.699951,1396.000000,1391.900024,1395.599976,153439,0.002010,0.000000,0.000000
3,RELIANCE.NS,2026-03-13 09:30:00+05:30,1395.599976,1397.099976,1394.000000,1395.900024,172155,0.000215,0.000000,0.000000
4,RELIANCE.NS,2026-03-13 09:35:00+05:30,1396.000000,1396.800049,1393.300049,1395.800049,159319,-0.000072,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
1390,INFY.NS,2026-03-23 10:05:00+05:30,1248.800049,1249.800049,1247.300049,1248.599976,94918,0.000160,1246.459998,1240.807998
1391,INFY.NS,2026-03-23 10:10:00+05:30,1248.599976,1249.300049,1246.699951,1248.500000,144359,-0.000080,1247.239996,1241.039998
1392,INFY.NS,2026-03-23 10:15:00+05:30,1248.500000,1250.400024,1247.800049,1249.199951,109070,0.000561,1248.164996,1241.303997
1393,INFY.NS,2026-03-23 10:20:00+05:30,1249.199951,1250.000000,1245.699951,1248.900024,86946,-0.000240,1249.019995,1241.571997
